In [9]:
import glob
import os
import random
from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.onnx
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

In [10]:
DATASET_DIR = "dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")
CHECKPOINT_PATH = "qrbitnet.pt"

QR_SIZE = 21

SPLIT_VALUE = 0.2
BATCH_SIZE = 64
EPOCHS = 2

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Number of workers:", NUM_WORKERS)

Device: cuda
GPU: NVIDIA GeForce GTX 1650
Number of workers: 0


In [12]:
class QRCodeDataset(Dataset):
    def __init__(self, images_dir: str, labels_dir: str) -> None:
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images = sorted(glob.glob(os.path.join(self.images_dir, "*.png")))

        items = []
        for image_path in self.images:
            base = os.path.splitext(os.path.basename(image_path))[0]
            label_path = os.path.join(self.labels_dir, f"{base}.txt")
            items.append((image_path, label_path))
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def read_label(self, path: str) -> torch.Tensor:
        with open(path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]

        matrix = []
        for line in lines:
            bits = line.split()
            row = [int(bit) for bit in bits]
            matrix.append(row)

        return torch.tensor(matrix, dtype=torch.float32)
        
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor, str]:
        image_path, label_path = self.items[index]

        image = Image.open(image_path).convert("L")
        array = np.array(image, dtype=np.float32)

        x = torch.from_numpy(array).unsqueeze(0)
        y = self.read_label(label_path)

        return x, y, os.path.basename(image_path)

In [13]:
class QRBitNet(nn.Module):
    def __init__(self, n: int = 21) -> None:
        super().__init__()
        self.n = n

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, n * n),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x)
        x = x.squeeze(-1).squeeze(-1)
        logits = self.fc(x)
        return logits.view(-1, self.n, self.n)

In [14]:
@torch.no_grad()
def compute_metrics(
    logits: torch.Tensor,
    y_true: torch.Tensor,
) -> Tuple[float, float]:
    probabilities = torch.sigmoid(logits)
    y_pred = (probabilities > 0.5).to(torch.float32)

    bit_accuracy = (y_pred == y_true).to(torch.float32).mean().item()

    correct_samples = (y_pred == y_true).view(y_true.size(0), -1).all(dim=1)
    exact_accuracy = correct_samples.to(torch.float32).mean().item()

    return bit_accuracy, exact_accuracy

In [15]:
def train_one_epoch(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    device: str,
    scaler: Optional[torch.cuda.amp.GradScaler] = None,
) -> float:
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Train", leave=False)
    
    for x, y, _ in progress_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optim.zero_grad(set_to_none=True)

        use_amp = (scaler is not None) and (device == "cuda")
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
            loss = loss_fn(logits, y)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
        else:
            loss.backward()
            optim.step()

        total_loss += loss.item() * x.size(0)
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_one_epoch(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    device: str,
) -> Tuple[float, float, float]:
    model.eval()
    total_loss = 0.0
    bit_accuracies, exact_accuracies = [], []
    progress_bar = tqdm(loader, desc="Val", leave=False)
    
    for x, y, _ in progress_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = loss_fn(logits, y)
        total_loss += loss.item() * x.size(0)

        bit_accuracy, exact_accuracy = compute_metrics(logits, y)
        bit_accuracies.append(bit_accuracy)
        exact_accuracies.append(exact_accuracy)

        progress_bar.set_postfix(bit=f"{bit_accuracy:.4f}", exact=f"{exact_accuracy:.4f}")

    return (
        total_loss / len(loader.dataset),
        float(np.mean(bit_accuracies)),
        float(np.mean(exact_accuracies)),
    )

In [18]:
dataset = QRCodeDataset(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
train_size = int((1 - SPLIT_VALUE) * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

model = QRBitNet(n=QR_SIZE).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

best_bit_accuracy = -1.0
best_exact_accuracy = -1.0
best_epoch = -1
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optim,
        loss_fn,
        DEVICE,
        scaler=scaler,
    )
    val_loss, val_bit_accuracy, val_exact_accuracy = eval_one_epoch(
        model,
        val_loader,
        loss_fn,
        DEVICE,
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Training loss: {train_loss:.4f} | "
        f"Validation loss: {val_loss:.4f} | "
        f"Validation bit accuracy: {val_bit_accuracy:.4f} | "
        f"Validation exact accuracy: {val_exact_accuracy:.4f}"
    )
    if val_bit_accuracy > best_bit_accuracy:
        best_bit_accuracy = val_bit_accuracy
        best_exact_accuracy = val_exact_accuracy
        best_epoch = epoch
        
        torch.save(
            {
                "model_state": model.state_dict(),
                "best_bit_accuracy": best_bit_accuracy,
                "best_exact_accuracy": best_exact_accuracy,
                "best_epoch": best_epoch,
            },
            CHECKPOINT_PATH,
        )
        
        print(
            f"New best model saved with validation bit accuracy: "
            f"{best_bit_accuracy:.4f}."
        )

print(
    f"\nTraining finished. The best validation bit accuracy was "
    f"{best_bit_accuracy:.4f}, achieved at epoch {best_epoch}.\n\n"
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()

x_sample, _, _ = dataset[0]
dummy_input = x_sample.unsqueeze(0).to(DEVICE)
_ = torch.onnx.export(
    model,
    dummy_input,
    "qrbitnet.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    dynamo=False
)

Epoch 01/2 | Training loss: 0.4377 | Validation loss: 0.3542 | Validation bit accuracy: 0.7800 | Validation exact accuracy: 0.0000
New best model saved with validation bit accuracy: 0.7800.


Epoch 02/2 | Training loss: 0.3549 | Validation loss: 0.3503 | Validation bit accuracy: 0.7800 | Validation exact accuracy: 0.0000
New best model saved with validation bit accuracy: 0.7800.

Training finished. The best validation bit accuracy was 0.7800, achieved at epoch 2.


